# Finetuning Qwen3.5

> Originally, adapted from [Qwen3_5_(0_8B)_Vision.ipynb](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb#scrollTo=gGFzmplrEy9I)

![Qwen3.5](https://qianwen-res.oss-accelerate.aliyuncs.com/logo_qwen3.5.png)

Qwen3.5 features the following enhancement:

- **Unified Vision-Language Foundation**: Early fusion training on multimodal tokens achieves cross-generational parity with Qwen3 and outperforms Qwen3-VL models across reasoning, coding, agents, and visual understanding benchmarks.
- **Efficient Hybrid Architecture**: Gated Delta Networks combined with sparse Mixture-of-Experts deliver high-throughput inference with minimal latency and cost overhead.
- **Scalable RL Generalization**: Reinforcement learning scaled across million-agent environments with progressively complex task distributions for robust real-world adaptability.
- **Global Linguistic Coverage**: Expanded support to 201 languages and dialects, enabling inclusive, worldwide deployment with nuanced cultural and regional understanding.
- **Next-Generation Training Infrastructure**: Near-100% multimodal training efficiency compared to text-only training and asynchronous RL frameworks supporting massive-scale agent scaffolds and environment orchestration.

In [ ]:
from unsloth import FastLanguageModel
# Force unsloth to be on top

import json
from pathlib import Path
import re

In [ ]:
MODEL_ID = "unsloth/Qwen3.5-9B"
MAX_NEW_TOKENS = 256
MAX_SEQUENCE_LENGTH = 4096 # SMT code can be up to 2048 tokens, so we need to ensure the model can handle that plus the question and answer.

# https://unsloth.ai/docs/models/qwen3.5#recommended-settings
ENABLE_THINKING = False
TEMPERATURE = 1.0
TOP_P = 0.95
TOP_K = 20
MIN_P = 0.0
PRESENCE_PENALTY = 1.5
REPETITION_PENALTY = 1.0

BASE_DIR = Path.cwd().parent
CATEGORY = "test"

# Input Files
INITIAL_STATE_PATH = BASE_DIR / f"submission_finetuning_{CATEGORY}_state.json"
SMT_FILE = BASE_DIR / f"smt_{CATEGORY}_state.json"

# Output Files
REFLECTION_STATE_PATH = BASE_DIR / f"submission_reflection_{CATEGORY}_state.json"
FINAL_SUBMISSION_PATH = BASE_DIR / f"submission_final_{CATEGORY}.json"

<a name="Data"></a>
### 🧪 Data Preparation

To convert our Sci-ImageMiner VQA data into the format required by Qwen2-VL (specifically for use with Unsloth), we need to restructure the data into a "messages" format.

The Qwen/Unsloth format expects a list of conversations where the image and the text prompt are bundled together in the user role, and the ground truth is in the assistant role, as follows:

```python
[
    { "role": "user",
    "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
    },
    { "role": "assistant",
    "content": [{"type": "text",  "text": A} ]
    },
]
```

In [ ]:
PROMPT_REWRITE = """
You are given an initial answer generated by a vision model, along with the SMT-LIB mathematical logic code executed to verify the figure's underlying data, and its resulting CVC5 solver output.

Question type: {question_type}
Question: {question}
Answer Type: {answer_type}

[INITIAL ANSWER]
{answer_cache}

[SMT-LIB CODE EXECUTED]
{code}

[SOLVER OUTPUT]
{output}

Evaluate the initial answer against the solver output.
- If the solver output contradicts the initial answer or provides a more mathematically/logically precise result, rewrite the answer to reflect the solver's findings.
- If the solver output confirms the initial answer or is irrelevant/failed, output the initial answer exactly as it is.

Strict Requirements:
1. Output plain text only, with no JSON, no code fences, and no surrounding explanatory text for the final answer.
2. Do NOT include any of the following in your final answer: the initial answer, the SMT-LIB code, the solver output, or any commentary on them. Your final answer should be a standalone response to the question.
3. You MUST adhere strictly to the format expected for a '{answer_type}' question. Apply the exact formatting rule corresponding to the question type below:
   - Yes/No: Output STRICTLY as "Yes" or "No" (title case).
   - Factoid: Stay concise and factual.
   - List: Output STRICTLY as comma-separated values (order-insensitive, no bullet points, no numbered lists).
   - Paragraph: Output STRICTLY as a paragraph containing at least 3 sentences providing an explanatory answer (no bullet points, no numbered lists).
4. You MUST enclose your final answer within <ANSWER>...</ANSWER> tags to clearly indicate the answer portion of your response.
"""

In [ ]:
with open(SMT_FILE, "r") as f:
    smt_data = json.load(f)

with open(INITIAL_STATE_PATH, "r") as f:
    initial_state = json.load(f)

<a name="Playground"></a>
### 🥼 Playground

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    load_in_4bit=True,
    max_seq_length=MAX_SEQUENCE_LENGTH,  # <--- Set this to match or exceed your expected context
    dtype=None,  # <--- Allows Unsloth to auto-detect the best compute dtype
)
FastLanguageModel.for_inference(model)

In [ ]:
target_sample_id = list(initial_state.keys())[0]
target_sub_fig = list(initial_state[target_sample_id].keys())[0]
target_q_obj = initial_state[target_sample_id][target_sub_fig][0]

print(f"Subfigure: {target_sub_fig}")
print(f"Targeting: {target_q_obj}")
print(f"Question: {target_q_obj['question']}")

In [ ]:
smt_entry = (
    smt_data.get(target_sample_id, {})
    .get(target_sub_fig, {})
    .get(target_q_obj["question"], {})
)

# Prepare the prompt
rewrite_prompt = PROMPT_REWRITE.format(
    question_type=target_q_obj.get("question_type", ""),
    question=target_q_obj.get("question", ""),
    answer_type=target_q_obj.get("answer_type", ""),
    answer_cache=target_q_obj.get("answer", ""),
    code=smt_entry.get("code", "N/A"),
    output=smt_entry.get("output", "N/A"),
)

print("--- CONSTRUCTED PROMPT ---")
print(rewrite_prompt)


In [ ]:
messages = [{"role": "user", "content": rewrite_prompt}]

input_text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    enable_thinking=ENABLE_THINKING,
)

inputs = tokenizer(text=input_text, return_tensors="pt").to("cuda")

output_ids = model.generate(
    **inputs,
    max_new_tokens=MAX_NEW_TOKENS,
    use_cache=True,
    temperature=TEMPERATURE,
    min_p=MIN_P,
    top_p=TOP_P,
    top_k=TOP_K,
    repetition_penalty=REPETITION_PENALTY,
)

full_response = tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[-1] :], skip_special_tokens=True
)

In [ ]:
initial = target_q_obj["answer"]

print("--- FULL MODEL RESPONSE ---")
print(full_response)
print("--- INITIAL ANSWER ---")
print(initial)

In [ ]:
# 1. Parsing
# Extract the text specifically inside the <ANSWER> tags requested in the prompt
match = re.search(r"<ANSWER>(.*?)</ANSWER>", full_response, re.DOTALL | re.IGNORECASE)

parsed_answer, hit_max_tokens = "", False
if match:
    parsed_answer = match.group(1).strip()
    hit_max_tokens = False  # We successfully found the closing tag


# 2. Logic Checks
is_empty = len(parsed_answer) == 0
is_rambling = len(parsed_answer) > (2 * len(initial))
is_too_short = len(initial) > 15 and len(parsed_answer) < (
    0.4 * len(initial)
)

# 3. Decision Tree
if is_empty or hit_max_tokens or is_rambling or is_too_short:
    reason = (
        "Empty"
        if is_empty
        else (
            "Hit Max/No Marker"
            if hit_max_tokens
            else ("Rambling" if is_rambling else "Too Short")
        )
    )
    print(f"❌ FALLBACK TRIGGERED. Reason: {reason}")
    final_answer = initial
else:
    print("✅ SUCCESS. Answer reflected.")
    final_answer = parsed_answer

print(f"\n--- FINAL ANSWER ---\n{final_answer}")
